In [ ]:
# SETUP: Install required packages
!pip install -q -U litellm python-dotenv

import os

# Option 1: Google Colab - Use userdata
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print("✓ Loaded API key from Colab secrets")
except ImportError:
    # Option 2: Local environment - Load from .env file
    from dotenv import load_dotenv
    load_dotenv()
    print("✓ Loaded API key from .env file")

# Verify API key is set
if not os.getenv('GEMINI_API_KEY'):
    print("⚠️  WARNING: GEMINI_API_KEY not found!")
    print("Please set it in Colab secrets or create a .env file with:")
    print("GEMINI_API_KEY=your_api_key_here")
else:
    print("✓ API key configured successfully")

print("\n=== LiteLLM Setup Complete ===")

In [ ]:
# BASIC: LiteLLM Approach - More flexible and production-ready
from typing import List, Dict
from litellm import completion
import os

def generate_response(messages: List[Dict], model: str = "gemini/gemini-3.6-flash") -> str:
    """
    Call LLM using LiteLLM - supports any model with OpenAI-compatible format.
    
    Args:
        messages: List of message dicts with 'role' and 'content'
        model: Model to use (default: gemini/gemini-3.6-flash)
               Can easily switch to: gpt-5.6-terra, claude-sonnet-5, etc.
    
    Returns:
        Response text from the model
    """
    try:
        response = completion(
            model=model,
            messages=messages,
            max_tokens=1024,  # Equivalent to max_output_tokens in native SDK
            temperature=0.7,  # Optional: control randomness (0.0 - 2.0)
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error calling LLM API: {e}"


# Example system prompt
SYSTEM_PROMPT = "You are an expert software engineer that prefers functional programming."

what_to_help_with = input("What do you need help with? ")

# Standard OpenAI-compatible message format
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": what_to_help_with},
]

# Get response
response_text = generate_response(messages)
print(response_text)

# BONUS: Multi-turn conversation example
# messages.append({"role": "assistant", "content": response_text})
# messages.append({"role": "user", "content": "Can you elaborate on that?"})
# print(generate_response(messages))

In [ ]:
# ADVANCED: LiteLLM with Streaming, Fallbacks, and Cost Tracking
from litellm import completion
from typing import List, Dict
import os

def generate_response_streaming(messages: List[Dict], model: str = "gemini/gemini-3.6-flash"):
    """
    Stream responses from LLM - great for real-time user experience
    """
    try:
        response = completion(
            model=model,
            messages=messages,
            stream=True,  # Enable streaming
            max_tokens=1024,
        )
        
        full_response = ""
        for chunk in response:
            content = chunk.choices[0].delta.content or ""
            full_response += content
            print(content, end="", flush=True)  # Print as it arrives
        
        print()  # New line at the end
        return full_response
    except Exception as e:
        return f"Error: {e}"


def generate_with_fallback(messages: List[Dict]):
    """
    Try multiple models with automatic fallback
    """
    models_to_try = [
        "gemini/gemini-3.6-flash",        # Primary: Fast and cheap
        "gemini/gemini-3.1-pro-preview",  # Fallback: More capable
    ]
    
    for model in models_to_try:
        try:
            print(f"Trying {model}...")
            response = completion(
                model=model,
                messages=messages,
                max_tokens=1024,
                num_retries=2,  # Retry up to 2 times on failure
            )
            
            # Access cost information
            if hasattr(response, '_hidden_params'):
                cost = response._hidden_params.get('response_cost', 'N/A')
                print(f"✓ Success! Cost: ${cost}")
            
            return response.choices[0].message.content
        except Exception as e:
            print(f"✗ Failed: {e}")
            continue
    
    return "All models failed"


# Example usage
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain quantum computing in simple terms"},
]

print("=== Streaming Response ===")
generate_response_streaming(messages)

print("\n=== With Fallback ===")
print(generate_with_fallback(messages))

In [ ]:
# PRODUCTION-READY: Using Router for Load Balancing & High Availability
from litellm import Router
from typing import List, Dict
import os

# Define multiple deployments for load balancing and reliability
model_list = [
    {
        "model_name": "gpt-3.5",  # Alias for your app
        "litellm_params": {
            "model": "gemini/gemini-3.6-flash",
            "api_key": os.getenv("GEMINI_API_KEY"),
        },
    },
    # Add more deployments for redundancy:
    # {
    #     "model_name": "gpt-3.5",
    #     "litellm_params": {
    #         "model": "gpt-5.6-luna",  # Fallback to OpenAI
    #         "api_key": os.getenv("OPENAI_API_KEY"),
    #     },
    # },
]

# Initialize router with reliability features
router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle",  # Load balance across deployments
    num_retries=3,                      # Retry failed requests
    timeout=30,                          # 30 second timeout
    # cache_responses=True,              # Enable caching (optional)
    # set_verbose=True,                  # Enable debug logs (optional)
)

# Use the router (same interface as completion)
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What are the benefits of using LiteLLM?"},
]

response = router.completion(
    model="gpt-3.5",  # Use the alias, not the actual model name
    messages=messages,
)

print(response.choices[0].message.content)
print(f"\nModel used: {response.model}")
print(f"Total tokens: {response.usage.total_tokens}")

In [ ]:
# COMPARISON: Test multiple models easily
from litellm import completion
from typing import List, Dict
import os

def compare_models(prompt: str, models: List[str]):
    """
    Compare responses from multiple models
    """
    messages = [{"role": "user", "content": prompt}]
    
    for model in models:
        print(f"\n{'='*60}")
        print(f"Model: {model}")
        print('='*60)
        
        try:
            response = completion(
                model=model,
                messages=messages,
                max_tokens=150,
            )
            print(response.choices[0].message.content)
            print(f"\nTokens: {response.usage.total_tokens}")
        except Exception as e:
            print(f"Error: {e}")

# Compare different Gemini models
compare_models(
    prompt="Explain the difference between machine learning and deep learning in 2 sentences.",
    models=[
        "gemini/gemini-3.6-flash",        # Fast & cheap
        "gemini/gemini-3.1-pro-preview",  # More capable
    ]
)